# Python como cliente SQL: SQLite y pandas

Un CSV guarda filas en un archivo. Una base SQL permite solicitar filas y columnas
mediante consultas, y guardar cambios con transacciones. Trabajaremos con un pequeño
catálogo de documentos para un asistente: título, tema y cantidad de páginas.

SQLite guarda la base en un archivo local; `sqlite3` viene con Python y no necesita
un servidor separado. El objetivo es conectar, consultar y recuperar datos desde
Python, no diseñar una base de datos.


In [1]:
import sqlite3
from contextlib import closing

import pandas as pd

from main import DB_PATH, prepare_database

prepare_database(DB_PATH)
print("Database:", DB_PATH)


Database: /Users/oscarnavarrete/Repos/programacion_ai/unidad-03-integracion-calidad/sesion-01-sql-nosql/outputs/documents.db


## 1. Conexión, consulta y resultados

`connect()` abre la conexión y `execute()` devuelve un cursor, desde el que recuperamos
los resultados. `fetchall()` devuelve una lista de tuplas; su orden coincide con las
columnas del `SELECT`. `ORDER BY` fija el orden de las filas.

`closing()` llama a `.close()` al salir de `with`, incluso si hay un error.
Usaremos `autocommit=False` para controlar cuándo se confirman los cambios.


In [2]:
with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    cursor = connection.execute("SELECT id, title, topic, pages FROM documents ORDER BY id")
    rows = cursor.fetchall()

for document_id, title, topic, pages in rows:
    print(document_id, title, topic, pages)


1 Python fundamentals python 28
2 Text embeddings retrieval 12
3 MCP tools integration 8
4 Graph search retrieval 16
5 NumPy arrays data 20
6 Pandas tables data 24


### Recuperar una sola fila

`fetchone()` devuelve una tupla o `None` si no hay coincidencias. El marcador `?`
recibe un valor de Python separado del texto SQL. `(4,)` es una tupla de un elemento;
la coma es necesaria. Evita construir consultas con f-strings o concatenaciones.


In [3]:
with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    document = connection.execute(
        "SELECT title, pages FROM documents WHERE id = ?", (4,)
    ).fetchone()
    missing_document = connection.execute(
        "SELECT title, pages FROM documents WHERE id = ?", (999,)
    ).fetchone()

print("Document:", document)
print("Missing document:", missing_document)


Document: ('Graph search', 16)
Missing document: None


### Filtrar desde Python

`WHERE` selecciona filas; `AND` combina condiciones. Los valores de `topic` y
`minimum_pages` se envían en una tupla, en el mismo orden que los marcadores.


In [4]:
topic = "retrieval"
minimum_pages = 10
query = """
    SELECT title, pages FROM documents
    WHERE topic = ? AND pages >= ?
    ORDER BY pages
"""
with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    selected = connection.execute(query, (topic, minimum_pages)).fetchall()
print(selected)


[('Text embeddings', 12), ('Graph search', 16)]


### Ejercicio 1 · Consulta parametrizada

Consulta los documentos del tema `data` con al menos 22 páginas. Devuelve título
y páginas en orden ascendente de páginas. Usa dos parámetros y `fetchall()`.


In [5]:
# Write your parameterized query here.


## 2. Guardar cambios con una transacción

Una transacción agrupa cambios: `commit` los confirma y `rollback` los revierte.
**`with connection` confirma al terminar sin errores o revierte si una excepción
sale del bloque; no cierra la conexión.** El `with closing(...)` exterior se encarga
del cierre. `autocommit=False` mantiene el control de transacciones activo.

Este ejemplo inserta un documento y lo consulta desde una nueva conexión.


In [6]:
with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    with connection:
        connection.execute(
            "INSERT OR REPLACE INTO documents (id, title, topic, pages) VALUES (?, ?, ?, ?)",
            (7, "Chunking strategies", "retrieval", 10),
        )

with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    saved_document = connection.execute(
        "SELECT title, pages FROM documents WHERE id = ?", (7,)
    ).fetchone()
print("Saved document:", saved_document)


Saved document: ('Chunking strategies', 10)


`INSERT OR REPLACE` permite repetir esta celda con el mismo ID: sustituye ese
registro. Para comprobar un rollback usaremos `UPDATE`, que cambia una fila existente,
y provocaremos un error antes de salir del bloque.


In [7]:
try:
    with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
        with connection:
            connection.execute("UPDATE documents SET pages = ? WHERE id = ?", (99, 1))
            raise ValueError("Cancel this change")
except ValueError as error:
    print(error)

with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    unchanged_pages = connection.execute(
        "SELECT pages FROM documents WHERE id = ?", (1,)
    ).fetchone()
print("Pages after rollback:", unchanged_pages)


Cancel this change
Pages after rollback: (28,)


### Ejercicio 2 · Actualización confirmada

Cambia las páginas del documento 7 a 14 usando parámetros y los dos bloques `with`.
Consulta ese documento desde una nueva conexión para verificar que se guardó.
Explica en una frase qué gestiona cada bloque.


In [8]:
# Update the document and verify it from a new connection here.


## 3. De SQL a un DataFrame

`pd.read_sql_query()` ejecuta un `SELECT` y devuelve un DataFrame. `params` pasa
los valores de los marcadores; las columnas vienen de la consulta. Después puedes
usar las operaciones de pandas que ya conoces. No hace falta pasar por CSV.


In [9]:
with closing(sqlite3.connect(DB_PATH, autocommit=False)) as connection:
    retrieval_documents = pd.read_sql_query(
        "SELECT id, title, pages FROM documents WHERE topic = ? ORDER BY id",
        connection,
        params=("retrieval",),
    )
print("Total pages:", retrieval_documents["pages"].sum())
retrieval_documents


Total pages: 38


,id,title,pages
0,2,Text embeddings,12
1,4,Graph search,16
2,7,Chunking strategies,10


### Ejercicio 3 · Resumen con pandas

Carga `title`, `topic` y `pages` de todos los documentos con `read_sql_query()`.
Con pandas, calcula por tema la cantidad de documentos y la media de páginas.
Entrega el DataFrame del resumen; puedes usar `groupby()` y `agg()`.


In [10]:
# Load the table and summarize it with pandas here.


## Continuar

Reiniciar y ejecutar todas las celdas restaura los seis documentos iniciales y
vuelve a aplicar los ejemplos y tus ejercicios. La base generada está en `outputs/`.
Consulta [PRACTICA.md](./PRACTICA.md) para entregar y continúa con la
[demostración de Neo4j](./u3_n2_neo4j.ipynb).

Fuentes: [sqlite3](https://docs.python.org/3/library/sqlite3.html),
[context managers de conexión](https://docs.python.org/3/library/sqlite3.html#how-to-use-the-connection-context-manager),
[closing](https://docs.python.org/3/library/contextlib.html#contextlib.closing),
[read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html).
